# FinGPT — Resume Backtest from Agent 1 CSV Cache

**Purpose**: re-run only Agent 2 (strategy selection) using a pre-computed Agent 1
result CSV so you never have to re-extract sentiment.

## Workflow
```
CSV (Agent 1 cache)
        │
        ▼
Reconstruct NewsFingerprint objects (ticker + sentiment from CSV)
        │
        ▼  [optional]
Merge with HuggingFace dataset for full article texts
        │
        ▼
Agent 2 — vLLM real logprobs  (A/B/C → BUY/HOLD/SELL)
   2 batched engine.generate() calls per batch of 10
        │
        ▼
Price fetch — yfinance daily bars  (1d interval, close-to-close return)
        │
        ▼
Metrics + new CSV
```

## Key fixes applied vs the original run
| Bug | Root cause | Fix |
|-----|-----------|-----|
| HOLD always wins | `"Strategy: "` prefix has strong LM prior for word "HOLD" | Score letters **A/B/C** instead — equal prior, context-discriminative |
| All price fetches fail | `interval="1wk"` over a 7-day window → 1 bar, need ≥2 | Switch to `interval="1d"` (5 trading days per window) |

In [ ]:
# ── 1. Install dependencies (Colab) ──────────────────────────────────────────
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

_pip("vllm")                                # vLLM inference engine
_pip("transformers", "accelerate")          # tokenizer + model loading
_pip("datasets")                            # HuggingFace datasets (optional merge)
_pip("yfinance", "pandas")                  # price fetching + data wrangling
_pip("python-dotenv", "pydantic")           # config + schema
print("Dependencies installed.")

In [ ]:
# ── 2. Mount Google Drive and set repo path ───────────────────────────────────
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab — skipping Drive mount.")

# ── Clone / locate the repo ───────────────────────────────────────────────────
REPO_DIR = "/content/FinGPT_Part2"          # change if you placed it elsewhere

if IN_COLAB and not os.path.exists(REPO_DIR):
    # Option A: clone from GitHub
    # !git clone https://github.com/<your-fork>/FinGPT_Part2.git {REPO_DIR}

    # Option B: copy from Drive
    DRIVE_REPO = "/content/drive/MyDrive/FinGPT_Part2"   # ← adjust path
    if os.path.exists(DRIVE_REPO):
        import shutil
        shutil.copytree(DRIVE_REPO, REPO_DIR)
    else:
        raise FileNotFoundError(
            f"Repo not found at {DRIVE_REPO}. "
            "Please clone it or adjust DRIVE_REPO above."
        )
elif not IN_COLAB:
    # Local development — point at the actual workspace
    REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), ""))
    if not os.path.exists(os.path.join(REPO_DIR, "config.py")):
        REPO_DIR = os.getcwd()              # fallback: current directory

# Add repo root to Python path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo dir : {REPO_DIR}")
print(f"sys.path : {sys.path[:3]}")

In [ ]:
# ── 3. Environment variables ───────────────────────────────────────────────────
import os

# Path to the DeepSeek-R1 / FinGPT model weights
os.environ["FINGPT_MODEL_PATH"] = "/content/drive/MyDrive/models/DeepSeek-R1-Distill-Llama-8B"

# Share one vLLM engine between both agents (saves GPU memory)
os.environ["SHARE_SINGLE_LLM_BETWEEN_AGENTS"] = "1"

# Use a FRESH yfinance cache so stale None values from the previous failed run
# don't shadow the corrected daily-interval fetches.
os.environ["FINGPT_YF_CACHE_PATH"] = "/tmp/yfinance_return_cache_resume.json"

# Softmax temperature for calibrating logprobs → probabilities
os.environ["FINGPT_CALIBRATION_T"] = "1.2"

# CoT token budget for Agent 2
os.environ["FINGPT_LOGITS_MAX_TOKENS"] = "1024"

print("Environment configured.")

In [ ]:
# ── 4. Load Agent 1 result CSV (the cache) ────────────────────────────────────
import ast
import pandas as pd

# ← Put the actual path to your CSV here
CSV_PATH = "/content/drive/MyDrive/backtest_20260501T015749Z.csv"

raw_df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(raw_df)} rows from CSV.")
print(f"Columns: {list(raw_df.columns)}")
print()

# Rows where Agent 1 succeeded (fingerprint_failed rows are skipped)
agent1_ok = raw_df[raw_df["skipped_reason"] != "fingerprint_failed"].copy()
print(f"Agent 1 successes: {len(agent1_ok)} / {len(raw_df)}")

# Parse sentiment_probabilities from string repr
def _parse_probs(val):
    if pd.isna(val) or not isinstance(val, str):
        return {"POSITIVE": 1/3, "NEGATIVE": 1/3, "NEUTRAL": 1/3}
    try:
        return ast.literal_eval(val)
    except Exception:
        return {"POSITIVE": 1/3, "NEGATIVE": 1/3, "NEUTRAL": 1/3}

agent1_ok["sentiment_probabilities_parsed"] = agent1_ok["sentiment_probabilities"].apply(_parse_probs)

display(agent1_ok[["ticker", "start_date", "end_date", "sentiment_label",
                    "sentiment_confidence", "skipped_reason"]].head(5))

In [ ]:
# ── 5. [Optional] Load HuggingFace dataset for full article texts ─────────────
#
# The CSV saves only the first 120 characters of each article.  Loading the
# original dataset and matching by (ticker, start_date) recovers the full text,
# which gives Agent 2 richer context.  Skip this cell if you prefer to use the
# truncated text or if the dataset is unavailable.

FULL_TEXT_MAP: dict[tuple, str] = {}   # (ticker, start_date) → full article_text

HF_DATASET_NAME = "Piyush5911/FinGPT_Sentiment"   # adjust if your dataset path differs

try:
    from datasets import load_dataset as hf_load_dataset
    from backtest.dataset_parser import build_backtest_rows, load_dataset as _load_ds

    # Try loading via the project's dataset_parser (handles local parquet / HF Hub)
    # If your dataset is a local CSV/parquet, pass its path instead of the HF name.
    hf_df = _load_ds(HF_DATASET_NAME)
    from backtest.dataset_parser import build_backtest_rows
    hf_rows = build_backtest_rows(hf_df)

    for r in hf_rows:
        key = (r["ticker"], r["start_date"])
        if key not in FULL_TEXT_MAP and r.get("article_text"):
            FULL_TEXT_MAP[key] = r["article_text"]

    print(f"Full-text map built: {len(FULL_TEXT_MAP)} entries.")
except Exception as exc:
    print(f"[optional] Could not load HF dataset: {exc}")
    print("Proceeding with 120-char truncated article texts from the CSV.")

In [ ]:
# ── 6. Reconstruct NewsFingerprint objects from CSV rows ──────────────────────
from agent1.schema import NewsFingerprint

_LABEL_TO_SCORE = {"POSITIVE": 1.0, "NEGATIVE": -1.0, "NEUTRAL": 0.0}

fingerprints: list[NewsFingerprint] = []
fp_row_idx: list[int] = []            # parallel index into agent1_ok

for idx, row in agent1_ok.iterrows():
    ticker      = str(row["ticker"])
    start_date  = str(row["start_date"])
    article_csv = str(row["article_text"]) if pd.notna(row["article_text"]) else ""

    # Prefer the full text from the HF dataset if available
    article_text = FULL_TEXT_MAP.get((ticker, start_date), article_csv)

    label = str(row.get("sentiment_label", "NEUTRAL"))
    if label not in _LABEL_TO_SCORE:
        label = "NEUTRAL"

    conf = float(row["sentiment_confidence"]) if pd.notna(row.get("sentiment_confidence")) else 0.5
    probs = row["sentiment_probabilities_parsed"]

    try:
        fp = NewsFingerprint(
            source="csv_cache",
            published_at=start_date,
            headline=article_text[:80],          # best approximation from truncated text
            companies_named=[ticker],
            event_keywords=[],
            sentiment_label=label,
            sentiment_score=_LABEL_TO_SCORE[label],
            sentiment_confidence=conf,
            sentiment_probabilities=probs,
            calibration_T=1.2,
            article_text=article_text,
        )
        fingerprints.append(fp)
        fp_row_idx.append(idx)
    except Exception as exc:
        print(f"[row {idx}] Fingerprint construction failed: {exc}")

print(f"Reconstructed {len(fingerprints)} fingerprints from {len(agent1_ok)} Agent 1 rows.")

In [ ]:
# ── 7. Load vLLM engine and inject into Agent 2 ───────────────────────────────
import os
from vllm import LLM
from transformers import AutoTokenizer
from agent2.reasoner import set_shared_vllm_engine, _ensure_chat_tokenizer
import agent2.reasoner as _a2

MODEL_PATH = os.environ["FINGPT_MODEL_PATH"]

print(f"Loading vLLM engine from: {MODEL_PATH}")
engine = LLM(
    model=MODEL_PATH,
    trust_remote_code=True,
    dtype="auto",
    gpu_memory_utilization=0.85,
    disable_log_stats=True,
    enforce_eager=True,
)
print("vLLM engine loaded.")

# Inject engine + tokenizer into Agent 2 module directly
set_shared_vllm_engine(engine)
_a2._chat_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Engine and tokenizer injected into Agent 2.")

In [ ]:
# ── 8. Run Agent 2 (strategy selection) in batches of 10 ─────────────────────
#
# Uses the fixed A/B/C scoring: letter tokens have balanced language-model
# priors, so the model's CoT reasoning actually drives the choice.

from agent2.reasoner import generate_signal_batch
from agent2.schema import TradingSignal
from typing import Optional

BATCH_SIZE = 10

signals: list[Optional[TradingSignal]] = []
total = len(fingerprints)

for batch_start in range(0, total, BATCH_SIZE):
    batch_fps = fingerprints[batch_start : batch_start + BATCH_SIZE]
    batch_end = batch_start + len(batch_fps)
    print(f"Agent 2 batch {batch_start + 1}–{batch_end} / {total} ...", end=" ", flush=True)
    try:
        batch_signals = generate_signal_batch(batch_fps)
    except Exception as exc:
        print(f"FAILED: {exc}")
        batch_signals = [None] * len(batch_fps)
    signals.extend(batch_signals)
    n_ok = sum(s is not None for s in batch_signals)
    print(f"ok={n_ok}/{len(batch_fps)}")

assert len(signals) == len(fingerprints)

# Quick distribution check
from collections import Counter
direction_counts = Counter(
    (s.direction if s else "None") for s in signals
)
print(f"\nSignal direction distribution: {dict(direction_counts)}")

In [ ]:
# ── 9. Fetch realized returns (fixed: daily interval) ─────────────────────────
#
# price_fetcher.py now uses interval="1d" (5 trading days per 7-day window)
# instead of "1wk" (which returned only 1 bar and always failed the len>=2 check).
# The FINGPT_YF_CACHE_PATH env var points to a fresh file so stale None
# entries from the previous run don't shadow the new correct fetches.

from backtest.price_fetcher import get_realized_return

realized_returns: list[Optional[float]] = []
fetch_ok = 0

for i, idx in enumerate(fp_row_idx):
    row = agent1_ok.loc[idx]
    ret = get_realized_return(
        ticker=str(row["ticker"]),
        start_date=str(row["start_date"]),
        end_date=str(row["end_date"]),
    )
    realized_returns.append(ret)
    if ret is not None:
        fetch_ok += 1
    if (i + 1) % 50 == 0:
        print(f"Price fetch progress: {i+1}/{len(fp_row_idx)} (ok={fetch_ok})")

print(f"\nPrice fetch complete: {fetch_ok}/{len(fp_row_idx)} returned a valid return.")

In [ ]:
# ── 10. Assemble result DataFrame and compute metrics ─────────────────────────
import math
from backtest.price_fetcher import direction_from_return

def _position_from_direction(direction: str) -> int:
    return {"long": 1, "short": -1}.get(direction, 0)

rows_out = []

for i, (df_idx, fp, sig, ret) in enumerate(
    zip(fp_row_idx, fingerprints, signals, realized_returns)
):
    orig = agent1_ok.loc[df_idx]
    record = {
        "ticker":        orig["ticker"],
        "start_date":    orig["start_date"],
        "end_date":      orig["end_date"],
        "article_text":  str(fp.article_text)[:120],
        "fingpt_label":  orig["fingpt_label"],
        # Agent 1 (from CSV cache)
        "sentiment_label":         fp.sentiment_label,
        "sentiment_confidence":    fp.sentiment_confidence,
        "sentiment_probabilities": str(fp.sentiment_probabilities),
        # Agent 2 (freshly generated)
        "signal_direction":        sig.direction        if sig else None,
        "signal_confidence":       sig.confidence       if sig else None,
        "signal_strategy_tag":     sig.strategy_tag     if sig else None,
        "signal_logits":           str(sig.signal_logits)       if sig else None,
        "signal_probabilities":    str(sig.signal_probabilities) if sig else None,
        # Backtest
        "realized_return":  ret,
        "position":         _position_from_direction(sig.direction) if sig else None,
        "strategy_return":  _position_from_direction(sig.direction) * ret
                            if sig and ret is not None else None,
        "skipped_reason":   "" if (sig and ret is not None)
                            else ("signal_failed" if not sig else "price_fetch_failed"),
    }
    rows_out.append(record)

results_df = pd.DataFrame(rows_out)

# ── Metrics ──────────────────────────────────────────────────────────────────
from backtest.backtester import compute_metrics
metrics = compute_metrics(results_df)

print("=" * 50)
print("BACKTEST METRICS")
print("=" * 50)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<28}: {v:.4f}")
    else:
        print(f"  {k:<28}: {v}")
print()

# Signal direction distribution
print("Signal direction breakdown:")
print(results_df["signal_direction"].value_counts(dropna=False).to_string())

In [ ]:
# ── 11. Save results to CSV ───────────────────────────────────────────────────
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT_DIR = os.path.join(REPO_DIR, "output")
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, f"backtest_resume_{timestamp}.csv")

results_df.to_csv(out_path, index=False)
print(f"Saved {len(results_df)} rows → {out_path}")

# Also copy to Drive if available
if IN_COLAB:
    drive_out = f"/content/drive/MyDrive/backtest_resume_{timestamp}.csv"
    results_df.to_csv(drive_out, index=False)
    print(f"Also saved to Drive → {drive_out}")